**Level 1 — File Handling**
- Task 1: Read: JSON file

In [0]:
df = spark.read.json("/Volumes/workspace/new/patient_json/patient_hw.json")
df.show()

+-----------+-----------+----------+------------+----------+
|bill_amount| department|patient_id|patient_name|visit_date|
+-----------+-----------+----------+------------+----------+
|       5000| Cardiology|         1|       Arjun|2024-01-01|
|       7000|  Neurology|         2|       Meera|2024-01-02|
|       4500|Orthopedics|         3|       Rahul|2024-01-03|
|       3000|Dermatology|         4|       Sneha|2024-01-04|
|       8000| Cardiology|         5|      Vikram|2024-01-05|
|       6500|  Neurology|         6|        Asha|2024-01-06|
|       5000|Orthopedics|         7|       Kiran|2024-01-07|
|       3500|Dermatology|         8|        Neha|2024-01-08|
|       7200| Cardiology|         9|        John|2024-01-09|
|       6800|  Neurology|        10|       Priya|2024-01-10|
|       9000| Cardiology|        11|       Akhil|2024-01-11|
|       4000|Orthopedics|        12|       Divya|2024-01-12|
|       2500|Dermatology|        13|       Rohit|2024-01-13|
|       7600|  Neurology

- Task 2: Check schema

In [0]:
df.printSchema()

root
 |-- bill_amount: long (nullable = true)
 |-- department: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- visit_date: string (nullable = true)



In [0]:
df.count()

25

**Level 2 — Validation**
- Task 3: Find: rows where bill_amount is null

In [0]:
from pyspark.sql.functions import *
df.filter(isnull(df.bill_amount)).show()

+-----------+----------+----------+------------+----------+
|bill_amount|department|patient_id|patient_name|visit_date|
+-----------+----------+----------+------------+----------+
|       NULL|Cardiology|        18|      Anjali|2024-01-18|
+-----------+----------+----------+------------+----------+



In [0]:
df = df.na.drop()
df.show()

+-----------+-----------+----------+------------+----------+
|bill_amount| department|patient_id|patient_name|visit_date|
+-----------+-----------+----------+------------+----------+
|       5000| Cardiology|         1|       Arjun|2024-01-01|
|       7000|  Neurology|         2|       Meera|2024-01-02|
|       4500|Orthopedics|         3|       Rahul|2024-01-03|
|       3000|Dermatology|         4|       Sneha|2024-01-04|
|       8000| Cardiology|         5|      Vikram|2024-01-05|
|       6500|  Neurology|         6|        Asha|2024-01-06|
|       5000|Orthopedics|         7|       Kiran|2024-01-07|
|       3500|Dermatology|         8|        Neha|2024-01-08|
|       7200| Cardiology|         9|        John|2024-01-09|
|       6800|  Neurology|        10|       Priya|2024-01-10|
|       9000| Cardiology|        11|       Akhil|2024-01-11|
|       4000|Orthopedics|        12|       Divya|2024-01-12|
|       2500|Dermatology|        13|       Rohit|2024-01-13|
|       7600|  Neurology

- Task 4: Find duplicate rows

In [0]:
dup = df.groupBy(df.columns).count().alias("count").filter("count>1")
dup.show()

+-----------+----------+----------+------------+----------+-----+
|bill_amount|department|patient_id|patient_name|visit_date|count|
+-----------+----------+----------+------------+----------+-----+
+-----------+----------+----------+------------+----------+-----+



- Task 5: Remove duplicates

In [0]:
df = df.dropDuplicates()
df.show()

+-----------+-----------+----------+------------+----------+
|bill_amount| department|patient_id|patient_name|visit_date|
+-----------+-----------+----------+------------+----------+
|       3500|Dermatology|         8|        Neha|2024-01-08|
|       7200| Cardiology|         9|        John|2024-01-09|
|       4500|Orthopedics|        23|       Rahul|2024-01-03|
|       4500|Orthopedics|         3|       Rahul|2024-01-03|
|       5000|Orthopedics|         7|       Kiran|2024-01-07|
|       6200| Cardiology|        15|      Sanjay|2024-01-15|
|       7000|  Neurology|         2|       Meera|2024-01-02|
|       3000|Dermatology|        24|       Sneha|2024-01-04|
|       8000| Cardiology|         5|      Vikram|2024-01-05|
|       6800|  Neurology|        10|       Priya|2024-01-10|
|       7000|  Neurology|        22|       Meera|2024-01-02|
|       2500|Dermatology|        13|       Rohit|2024-01-13|
|       2800|Dermatology|        17|       Kevin|2024-01-17|
|       7100|  Neurology

In [0]:
df.count()

24

In [0]:
df = df.withColumn("bill_amount", df.bill_amount.cast("float"))
df.printSchema()

root
 |-- bill_amount: float (nullable = true)
 |-- department: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- visit_date: string (nullable = true)



# Level 3 — Transformations
- Task 6: Create: billing_category
    - <5000 → Low
    - 5000–7000 → Medium
    - 7000 → High

In [0]:
df = df.withColumn("billing_category", when(df.bill_amount < 5000, "Low"). when((df.bill_amount >= 5000) & (df.bill_amount < 7000), "Medium").otherwise("High"))
df.show()

+-----------+-----------+----------+------------+----------+----------------+
|bill_amount| department|patient_id|patient_name|visit_date|billing_category|
+-----------+-----------+----------+------------+----------+----------------+
|     3500.0|Dermatology|         8|        Neha|2024-01-08|             Low|
|     7200.0| Cardiology|         9|        John|2024-01-09|            High|
|     4500.0|Orthopedics|        23|       Rahul|2024-01-03|             Low|
|     4500.0|Orthopedics|         3|       Rahul|2024-01-03|             Low|
|     5000.0|Orthopedics|         7|       Kiran|2024-01-07|          Medium|
|     6200.0| Cardiology|        15|      Sanjay|2024-01-15|          Medium|
|     7000.0|  Neurology|         2|       Meera|2024-01-02|            High|
|     3000.0|Dermatology|        24|       Sneha|2024-01-04|             Low|
|     8000.0| Cardiology|         5|      Vikram|2024-01-05|            High|
|     6800.0|  Neurology|        10|       Priya|2024-01-10|    

- Task 7: Convert: visit_date → proper date type

In [0]:
df = df.withColumn("visit_date", to_date(df.visit_date))
df.show()

+-----------+-----------+----------+------------+----------+----------------+
|bill_amount| department|patient_id|patient_name|visit_date|billing_category|
+-----------+-----------+----------+------------+----------+----------------+
|     3500.0|Dermatology|         8|        Neha|2024-01-08|             Low|
|     7200.0| Cardiology|         9|        John|2024-01-09|            High|
|     4500.0|Orthopedics|        23|       Rahul|2024-01-03|             Low|
|     4500.0|Orthopedics|         3|       Rahul|2024-01-03|             Low|
|     5000.0|Orthopedics|         7|       Kiran|2024-01-07|          Medium|
|     6200.0| Cardiology|        15|      Sanjay|2024-01-15|          Medium|
|     7000.0|  Neurology|         2|       Meera|2024-01-02|            High|
|     3000.0|Dermatology|        24|       Sneha|2024-01-04|             Low|
|     8000.0| Cardiology|         5|      Vikram|2024-01-05|            High|
|     6800.0|  Neurology|        10|       Priya|2024-01-10|    

- Task 8: Extract: month from visit_date

In [0]:
df = df.withColumn("Month", month(df.visit_date))
df.show()

+-----------+-----------+----------+------------+----------+----------------+-----+
|bill_amount| department|patient_id|patient_name|visit_date|billing_category|Month|
+-----------+-----------+----------+------------+----------+----------------+-----+
|     3500.0|Dermatology|         8|        Neha|2024-01-08|             Low|    1|
|     7200.0| Cardiology|         9|        John|2024-01-09|            High|    1|
|     4500.0|Orthopedics|        23|       Rahul|2024-01-03|             Low|    1|
|     4500.0|Orthopedics|         3|       Rahul|2024-01-03|             Low|    1|
|     5000.0|Orthopedics|         7|       Kiran|2024-01-07|          Medium|    1|
|     6200.0| Cardiology|        15|      Sanjay|2024-01-15|          Medium|    1|
|     7000.0|  Neurology|         2|       Meera|2024-01-02|            High|    1|
|     3000.0|Dermatology|        24|       Sneha|2024-01-04|             Low|    1|
|     8000.0| Cardiology|         5|      Vikram|2024-01-05|            High

**Level 4 — Analysis**
- Task 9: Find: total billing per department

In [0]:
df.groupBy("department").agg(sum("bill_amount").alias("total_dep_revenue")).show()

+-----------+-----------------+
| department|total_dep_revenue|
+-----------+-----------------+
|Dermatology|          14800.0|
| Cardiology|          48400.0|
|Orthopedics|          27400.0|
|  Neurology|          42000.0|
+-----------+-----------------+



- Task 10: Find: average billing per department

In [0]:
df.groupBy("department").agg(avg("bill_amount").alias("avg_dep_revenue")).show()

+-----------+-----------------+
| department|  avg_dep_revenue|
+-----------+-----------------+
|Dermatology|           2960.0|
| Cardiology|6914.285714285715|
|Orthopedics|4566.666666666667|
|  Neurology|           7000.0|
+-----------+-----------------+



- Task 11: Find: department with highest billing

In [0]:
df.groupBy("department").agg(sum("bill_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)


+----------+-------------+
|department|total_revenue|
+----------+-------------+
|Cardiology|      48400.0|
+----------+-------------+
only showing top 1 row


**Level 5 — ETL Flow**
- Task 12: Build complete flow:
    - read
    - validate
    - clean
    - transform
    - analyze

In [0]:
from pyspark.sql.functions import *

#read the json file
print("Reading the json file: ")
df = spark.read.json("/Volumes/workspace/new/patient_json/patient_hw.json")

#validate the file
print("Table validation: ")
df.printSchema()
df.count()
print("Null value row count: ",df.filter(isnull(df.bill_amount)).count())
df.filter(isnull(df.bill_amount)).show()
print("duplicate row count: ", df.groupBy(df.columns).count().filter("count > 1").count())
df.groupBy(df.columns).count().alias("count").filter("count>1").show()

#clean the data
print("Cleaning up: ")
df = df.na.drop()
df = df.dropDuplicates()
df = df.withColumn("bill_amount", df.bill_amount.cast("float"))
df = df.withColumn("visit_date", to_date(df.visit_date))

df.show()
df.printSchema()

#Transform the data
print("Transforming: ")
df = df.withColumn(("Month"),month(df.visit_date))
df.show()

#Analysis
print("\n Analysis: ")
df.groupBy("department").agg(sum("bill_amount").alias("total_dep_revenue")).show()
df.groupBy("department").agg(avg("bill_amount").alias("avg_dep_revenue")).show()
df.groupBy("department").agg(sum("bill_amount").alias("total_dep_revenue")).show()

Reading the json file: 
Table validation: 
root
 |-- bill_amount: long (nullable = true)
 |-- department: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- visit_date: string (nullable = true)

Null value row count:  1
+-----------+----------+----------+------------+----------+
|bill_amount|department|patient_id|patient_name|visit_date|
+-----------+----------+----------+------------+----------+
|       NULL|Cardiology|        18|      Anjali|2024-01-18|
+-----------+----------+----------+------------+----------+

duplicate row count:  0
+-----------+----------+----------+------------+----------+-----+
|bill_amount|department|patient_id|patient_name|visit_date|count|
+-----------+----------+----------+------------+----------+-----+
+-----------+----------+----------+------------+----------+-----+

Cleaning up: 
+-----------+-----------+----------+------------+----------+
|bill_amount| department|patient_id|patient_name|vis

#  Challenge Tasks
- Task 13: Find: month with highest total billing

In [0]:
df.groupBy("Month").agg(sum("bill_amount").alias("total_dep_revenue")).orderBy(desc("total_dep_revenue")).show()

+-----+-----------------+
|Month|total_dep_revenue|
+-----+-----------------+
|    1|         132600.0|
+-----+-----------------+

